### Recommendation : real music, not just OSTs

Goals : blend a few games into one mood and recommend actual songs from real artists for it, instead of handing back yet another game soundtrack.

In notebook 04 I got a similarity recommender working, but it had two limits : it looked at **one game at a time**, and it could only ever return **other OSTs**, since OSTs are the only thing I collected. This notebook tackles both : I blend several games into a single mood, and I search a big pool of **real music** instead of soundtracks. The blocker for that second part is that I need the same 8 audio features for normal music, and my OST features came from RapidAPI.

> Why I'm **not** using RapidAPI this time :
> 1. RapidAPI works, but I'd have to fetch every single track one by one, and that's **very long** (hundreds of calls, retries, rate limits)... not worth it here.
> 2. I don't actually need the freshest songs of the month for this, I just want a big, varied pool of music carrying the same 8 features.
> 3. Spotify's audio features were public for years before they deprecated the endpoint, so tons of them are already sitting in open datasets.
>
> So instead of fetching anything, I just grab a ready-made Spotify audio-features dataset (114k songs). Same 8 features, zero API calls, instant.

> I'm pulling the dataset straight from Kaggle with `kagglehub` (it caches it locally so I only download it once).

> One gotcha I ran into : this dataset lists the same song once **per genre** (and sometimes for a couple of different releases), so a track like *Freaks* by Surf Curse shows up 6 times. If I don't handle it, the exact same song comes back several times in my recommendations. So right after loading I keep **one row per song** (the most popular version) before doing anything else.

In [26]:
import os
import numpy as np
import pandas as pd

feats = ["danceability","energy","valence","acousticness","instrumentalness","liveness","speechiness","tempo"]

# public Spotify audio-features dataset, NO RapidAPI fetching
try:
    import kagglehub
    path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset")
    songs = pd.read_csv(os.path.join(path, "dataset.csv"))
except Exception as e:
    print("kagglehub not available, reading cached copy instead:", e)
    songs = pd.read_csv("../data/external/spotify_tracks_hf.csv")

songs = songs.dropna(subset=feats)
# the dataset lists the same song once PER genre (and sometimes for a few releases),
# so I keep a single row per song (the most popular version) to avoid recommending duplicates
songs = (songs.sort_values("popularity", ascending=False)
              .drop_duplicates(subset=["track_name", "artists"])
              .reset_index(drop=True))
print(songs.shape[0], "unique songs |", songs["track_genre"].nunique(), "genres")
songs[["track_name","artists","track_genre","danceability","energy","valence","tempo"]].head()

kagglehub not available, reading cached copy instead: No module named 'kagglehub'
81344 unique songs | 114 genres


,track_name,artists,track_genre,danceability,energy,valence,tempo
0,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,dance,0.714,0.472,0.238,131.121
1,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,hip-hop,0.621,0.782,0.550,128.033
2,I'm Good (Blue),David Guetta;Bebe Rexha,pop,0.561,0.965,0.304,128.040
3,La Bachata,Manuel Turizo,reggaeton,0.835,0.679,0.850,124.980
4,Me Porto Bonito,Bad Bunny;Chencho Corleone,latino,0.911,0.712,0.425,92.005


### The one catch : the two sources aren't on the same scale

> There's a trap when mixing two sources. My OST features came from **RapidAPI on a 0–100 scale**, while this Spotify dataset is on the classic **0–1 scale**. If I blindly stack them together, the cosine similarity would be completely dominated by the OST numbers being 100x bigger, and the recommendations would be garbage. So before anything I have to put them in the same units.

In [27]:
ost = pd.read_csv("../data/rapidapi_features_clean.csv").dropna(subset=feats).copy()

# see the problem : same features, very different scale
pd.DataFrame({
    "OST (RapidAPI, 0-100)": ost[feats].mean(),
    "Songs (Spotify, 0-1)": songs[feats].mean(),
}).round(2)

,"OST (RapidAPI, 0-100)","Songs (Spotify, 0-1)"
danceability,45.21,0.56
energy,45.76,0.63
valence,31.73,0.46
acousticness,44.20,0.33
instrumentalness,65.40,0.18
liveness,17.29,0.22
speechiness,5.93,0.09
tempo,119.77,122.13


> The fix is easy : everything except `tempo` (which is in BPM in both) is basically a percentage, so I divide the OST features by 100. Then I fit **one single** MinMax scaler on the two datasets together, so my games and my songs end up living in the exact same feature space, that's the only way cosine similarity between them means something.

In [28]:
from sklearn.preprocessing import MinMaxScaler

pct = [f for f in feats if f != "tempo"] # everything except tempo is a 0-100 percentage
ost[pct] = ost[pct] / 100.0 # bring RapidAPI onto Spotify's 0-1 scale

# one scaler fit on BOTH so OSTs and songs share the exact same space
scaler = MinMaxScaler().fit(pd.concat([ost[feats], songs[feats]]))
ost_s = scaler.transform(ost[feats])
songs_s = scaler.transform(songs[feats])
print("scaled OSTs:", ost_s.shape, "| scaled songs:", songs_s.shape)

scaled OSTs: (753, 8) | scaled songs: (81344, 8)


### Blending games into a meta-mood

> This is the new part : instead of querying a single game like in notebook 04, I blend several. A game's **signature** is just the mean of its (now re-scaled) tracks over the 8 features, and a **meta-mood** is a weighted average of several signatures (so I can say 60% one game, 40% another). Then I rank the 114k real songs against that blended mood.

In [29]:
def game_signature(game):
    return ost_s[(ost["game_title"] == game).values].mean(axis=0)

def mood_centroid(weights):
    total = sum(weights.values())
    return sum((w / total) * game_signature(g) for g, w in weights.items())

> I score every real song by cosine similarity to the mood, keep the closest ones, then re-rank with **MMR** (same diversity trick as in notebook 04) so I don't get 15 nearly identical lo-fi tracks in a row. I also add a `min_popularity` knob, when it's 0 I get hidden gems, when I push it up I get songs I'd actually recognize, mostly pop.

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_songs(weights, top_n=15, min_popularity=10, lam=0.7, pool_size=250):
    mood = mood_centroid(weights)
    mask = (songs["popularity"] >= min_popularity).values
    pool = songs[mask].reset_index(drop=True)
    pool_s = songs_s[mask]

    # 1) keep the closest songs to the mood
    sims = cosine_similarity(pool_s, mood.reshape(1, -1)).ravel()
    order = sims.argsort()[::-1][:pool_size]
    pool, pool_s, sims = pool.iloc[order].reset_index(drop=True), pool_s[order], sims[order]

    # 2) MMR re-rank : relevance (lam) vs diversity (1 - lam)
    selected, idxs = [], list(range(len(pool_s)))
    while len(selected) < top_n and idxs:
        if not selected:
            best = idxs[int(np.argmax(sims[idxs]))]
        else:
            rest = np.array(idxs)
            redundancy = cosine_similarity(pool_s[rest], pool_s[selected]).max(axis=1)
            best = int(rest[int(np.argmax(lam * sims[rest] - (1 - lam) * redundancy))])
        selected.append(best); idxs.remove(int(best))

    out = pool.iloc[selected].copy()
    out["sim"] = sims[selected]
    return out

blend = {"Genshin Impact": 0.6, "Life is Strange 2": 0.4}
recommend_songs(blend, top_n=15)[["track_name","artists","track_genre","popularity","sim"]]

,track_name,artists,track_genre,popularity,sim
0,Right Where It Belongs,Nine Inch Nails,industrial,42,0.998974
1,Love the Way You Lie (Slowed + Reverb),fenekot,chill,47,0.997274
2,Stranger To Yourself,Loving,garage,53,0.995349
3,Mystery of Love,Sufjan Stevens,songwriter,71,0.994936
6,"Libertango (from ""Cuatro Tangos"") - Latino Gol...",Miloš Karadaglić;Ksenija Sidorova;Studioorches...,guitar,22,0.993517
5,Saware - Lofi Mix,It's DPK;Nidhi Hegde,indian,59,0.993571
7,Manners Maketh Man,Henry Jackman;Matthew Margeson,british,45,0.993322
4,Love Brought Weight,Old Sea Brigade,folk,64,0.993615
9,Raat Raazi,Prateek Kuhad,folk,41,0.992641
16,"Vinna – Spotify Studio Recording (från ""De fria"")",Moonica Mac,swedish,35,0.990288


> And there we go, **real artists** now. The cozy + sci-fi blend pulls chill, atmospheric, study-ish tracks, which makes complete sense : the two OSTs I picked are very instrumental, so the recommender naturally leans toward instrumental real music. This is exactly the bridge between gaming and music I wanted from the very start of the project.

> Same blend, but this time I only keep the more mainstream songs so I get stuff I'd actually recognize :

In [31]:
recommend_songs(blend, top_n=15, min_popularity=85)[["track_name","artists","track_genre","popularity","sim"]]

,track_name,artists,track_genre,popularity,sim
0,everything i wanted,Billie Eilish,electro,86,0.968869
1,Glimpse of Us,Joji,pop,94,0.932253
2,traitor,Olivia Rodrigo,pop,88,0.931898
3,lovely (with Khalid),Billie Eilish;Khalid,pop,89,0.929573
4,All of Me,John Legend,soul,85,0.927795
6,Bored,Billie Eilish,electro,85,0.923065
5,drivers license,Olivia Rodrigo,pop,88,0.923730
7,Happier Than Ever,Billie Eilish,pop,88,0.919759
8,happier,Olivia Rodrigo,pop,86,0.914724
11,Night Changes,One Direction,pop,88,0.908726


### Pick your own games

> Instead of hardcoding the blend, here's a little selector : it lists **all the games** I have and I just pick the ones I'm in the mood for (ctrl / cmd-click to choose several). They get blended with equal weight, and I can slide the minimum popularity and how many tracks I want. I hit **Run** and it builds the meta-mood and recommends real songs on the fly.

In [36]:
import ipywidgets as widgets
from IPython.display import display

all_games = sorted(ost["game_title"].unique())

game_picker = widgets.SelectMultiple(
    options=all_games,
    value=("Genshin Impact", "Life is Strange 2"),
    rows=9,
    description="Games",
    layout=widgets.Layout(width="70%"),
)
pop_slider = widgets.IntSlider(value=10, min=0, max=100, step=5, description="min pop")
n_slider = widgets.IntSlider(value=15, min=5, max=30, step=1, description="top N")

@widgets.interact_manual(games=game_picker, min_pop=pop_slider, top_n=n_slider)
def pick_and_recommend(games, min_pop, top_n):
    if not games:
        print("Pick at least one game (ctrl / cmd-click to select several).")
        return
    blend = {g: 1 for g in games} # equal weight per chosen game
    print("Blending:", ", ".join(games))
    out = recommend_songs(blend, top_n=top_n, min_popularity=min_pop)
    display(out[["track_name", "artists", "track_genre", "popularity", "sim"]])

interactive(children=(SelectMultiple(description='Games', index=(12, 15), layout=Layout(width='70%'), options=…

### Export the real-music album

In [37]:
def export_m3u(tracks, path):
    lines = ["#EXTM3U"]
    for _, r in tracks.iterrows():
        lines.append(f"#EXTINF:-1,{r['artists']} - {r['track_name']}")
        lines.append(f"https://open.spotify.com/track/{r['track_id']}")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")

final = recommend_songs(blend, top_n=50, min_popularity=80)
final.to_csv("../data/metamood_real_songs.csv", index=False)
export_m3u(final, "../data/metamood_real_songs.m3u8")
print("saved", len(final), "real songs\n")
print(open("../data/metamood_real_songs.m3u8").read())

saved 50 real songs

#EXTM3U
#EXTINF:-1,Sufjan Stevens - Fourth of July
https://open.spotify.com/track/5Qnrgqy1pAm9GyNQOgyVFz
#EXTINF:-1,Yot Club - YKWIM?
https://open.spotify.com/track/2vWBUC9djv6BtiGlmKiQaH
#EXTINF:-1,Billie Eilish - everything i wanted
https://open.spotify.com/track/3ZCTVFBt2Brf31RLEnCkWJ
#EXTINF:-1,JVKE - golden hour
https://open.spotify.com/track/5odlY52u43F5BjByhxg7wg
#EXTINF:-1,Tom Rosenthal - Lights Are On
https://open.spotify.com/track/4IhTXiZLKATmwhMZIb1GQN
#EXTINF:-1,Billie Eilish - ocean eyes
https://open.spotify.com/track/7hDVYcQq6MxkdJGweuCtl9
#EXTINF:-1,Liana Flores - rises the moon
https://open.spotify.com/track/51Grh1RyUDcMBbpuyUIUHI
#EXTINF:-1,Lord Huron - The Night We Met
https://open.spotify.com/track/3hRV0jL3vUpRrcy398teAU
#EXTINF:-1,Billie Eilish;Khalid - lovely (with Khalid)
https://open.spotify.com/track/0u2P5u6lvoDfwTYjAADbn4
#EXTINF:-1,Joji - Glimpse of Us
https://open.spotify.com/track/6xGruZOHLs39ZbVccQTuPZ
#EXTINF:-1,Olivia Rodrigo - traito

### Wrap up

> MetaTune can now recommend **real songs from real artists** for any blend of games, not just other OSTs, all from the same meta-mood. And I did it without spending hours fetching from RapidAPI, by reusing a public Spotify audio-features dataset.
>
> Next steps :
> 1. finally wire the Spotify export so the playlist lands straight in my library
> 2. the radar chart from the roadmap, to actually *see* a song overlap the mood